# Figure 4: heart-rate sensitivity

The cell below is tagged `#| label: fig-heart-rate-output`, so `05-issues.md` embeds its output with `:::{figure} #fig-heart-rate-output`. The simulation is in `cardiac_sim.py` in this folder.

In [ ]:
#| label: fig-heart-rate-output
# Figure 4: error of the fitted T1 as a function of heart rate.
# The simulation itself is in cardiac_sim.py (shared with Figures 3 and 5).
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import warnings, logging
# Kaleido (the PNG export for the PDF) can print harmless clean-up warnings on Windows.
warnings.filterwarnings("ignore", message=".*temporary directory.*")
for name in ("choreographer", "kaleido", "logistro"):
    logging.getLogger(name).setLevel(logging.CRITICAL)
import os, sys
sys.path.append(os.path.join(os.getcwd(), "notebooks"))  # find cardiac_sim.py also when run from the book folder
from cardiac_sim import SEQUENCES, simulate, fit

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

TISSUES = {"myocardium (T1 = 1000 ms)": 1000.0, "blood (T1 = 1600 ms)": 1600.0,
           "myocardium after contrast (T1 = 400 ms)": 400.0}
HEART_RATES = np.arange(40, 122, 2)              # beats per minute
COLORS = {"MOLLI": "royalblue", "ShMOLLI": "seagreen", "SASHA": "darkorange"}

fig = go.Figure()
for i, (tissue, T1) in enumerate(TISSUES.items()):
    for seq in SEQUENCES:
        errors = []
        for hr in HEART_RATES:
            RR = 60000.0 / hr                    # time between two heartbeats [ms]
            _, _, _, samples = simulate(seq, T1, RR)
            T1_fit = fit(seq, samples, RR)[2]
            errors.append(100 * (T1_fit - T1) / T1)
        fig.add_trace(go.Scatter(x=HEART_RATES, y=errors, mode="lines+markers",
                                 line=dict(color=COLORS[seq]), marker=dict(size=5),
                                 name=seq, visible=(i == 0)))

buttons = []
for i, tissue in enumerate(TISSUES):
    visible = [False] * (len(TISSUES) * len(SEQUENCES))
    visible[i * len(SEQUENCES):(i + 1) * len(SEQUENCES)] = [True] * len(SEQUENCES)
    buttons.append(dict(label=tissue, method="update", args=[{"visible": visible}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", x=0, xanchor="left", y=1.15, yanchor="top")],
    height=540, margin=dict(t=80, b=140), template="plotly_white",
    xaxis_title="Heart rate [beats per minute]", yaxis_title="Error of the fitted T1 [%]",
    yaxis_range=[-20, 3],
    # legend at the very bottom of the figure, with space below the x-axis title
    legend=dict(orientation="h", yref="container", yanchor="bottom", y=0.01, x=0),
)
fig
